# One 256×256 scene: its inputs and its six routes

A PASSAGE scene is a three-channel tensor and six stored routes. This notebook opens one validation
scene of the 256×256 configuration and shows each part in turn: the input channels a
planner receives, the six reference routes on one terrain scale, and the length and cost of each
route. It also re-costs every route with the evaluator's own cost function, so you can check that
each stored cost belongs to its stored route. The scene is the first of the split whose six routes all
span at least a quarter of the grid side and whose obstacles cover at least 5 % of the cells; this
rule, fixed in advance, avoids endpoints that sit next to each other and scenes without obstacles.

**Run it yourself.** From a checkout of the [code repository](https://github.com/ThalesGroup/passage), start Jupyter with `uv run --frozen jupyter lab`, open this notebook and point `dataset_path` to a local copy of [the dataset](https://huggingface.co/datasets/thalesgroup/passage). With `dataset_path = None`, rows are read from the Hub at the pinned `dataset_revision` instead. It reads one row and runs in a few seconds.

In [ ]:
resolution = 256
split = "validation"
sample_id = None  # None applies the selection rule above; a missing ID raises an error
output_dir = "notebook-artifacts/256x256/display"
dataset_path = "dataset"  # local copy of the dataset; None reads the Hub at dataset_revision
dataset_repo = "thalesgroup/passage"
dataset_revision = "2ab44d899652c7fb99c425488e84a7d565963547"  # verified PASSAGE v1.0 upload
project_path = None  # checkout root; None searches upwards from the working directory

In [ ]:
from pathlib import Path
import json  # noqa: F401
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np  # noqa: F401
import pandas as pd
from IPython.display import Markdown, display

# Locate the code checkout (pyproject.toml next to src/passage) without changing directory.
project_root = Path(project_path).resolve() if project_path else next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "src/passage").is_dir()), Path.cwd())
for extra in (project_root / "src", project_root / "scripts"):
    if extra.is_dir() and str(extra) not in sys.path:
        sys.path.insert(0, str(extra))

from passage.notebook_reports import make_source, resolve_path, save_figure, write_report  # noqa: E402, F401

source = make_source(dataset_path, dataset_repo=dataset_repo, dataset_revision=dataset_revision,
                     project_root=project_root)
output_path = resolve_path(output_dir, project_root)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 30)


def relative(path):
    """Show saved files relative to the checkout, never as machine paths."""
    path = Path(path)
    try:
        return path.resolve().relative_to(project_root.resolve()).as_posix()
    except ValueError:
        return path.name


def show(frame, caption=None, floatfmt="{:,.3f}"):
    """Display a table with readable numbers and an optional caption."""
    if caption:
        display(Markdown(f"*{caption}*"))
    floats = list(frame.select_dtypes("float").columns)
    ints = list(frame.select_dtypes("integer").columns)
    others = [c for c in frame.columns if c not in floats and c not in ints]
    styler = frame.style.hide(axis="index")
    for columns, fmt in ((floats, floatfmt), (ints, "{:,}"), (others, None)):
        if columns:
            styler = styler.format(fmt, subset=columns, na_rep="—")
    display(styler)


# One chart style for every PASSAGE notebook: recessive axes, light grid, left-aligned titles.
INK, MUTED, GRID = "#243537", "#5c6765", "#e3e6e3"
OBJECTIVE_COLORS = {"elevation": "#1767ac", "slope_uphill": "#d56418", "energy": "#a32b8e"}
OBJECTIVE_LABELS = {"elevation": "Elevation", "slope_uphill": "Slope (uphill)", "energy": "Energy"}
SPLIT_COLORS = {"train": "#1f8a70", "calibration": "#b07d1a", "validation": "#3a62b5", "test": "#c0392b"}
REGIME_COLORS = {"free": "#238b45", "obstacles": "#1767ac"}
plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 9, "axes.edgecolor": "#8a9290", "axes.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "axes.grid.axis": "y",
    "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True, "axes.titlelocation": "left",
    "axes.titlesize": 10.5, "axes.titleweight": "semibold", "axes.titlecolor": INK, "axes.labelcolor": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "legend.frameon": False, "lines.linewidth": 1.6,
    "figure.dpi": 110, "savefig.dpi": 140, "pdf.fonttype": 42, "ps.fonttype": 42,
})


def figure_title(fig, title, subtitle=None):
    """Left-aligned title and subtitle in a band reserved at the top of the figure."""
    height = fig.get_figheight()
    fig.suptitle(title, x=0.01, y=1 - 0.1 / height, ha="left", va="top", fontsize=12, fontweight="semibold", color=INK)
    if subtitle:
        fig.text(0.01, 1 - 0.4 / height, subtitle, ha="left", va="top", fontsize=8.5, color=MUTED)
    fig.subplots_adjust(top=min(fig.subplotpars.top, 1 - 0.95 / height))


# The shared trajectory style of the paper: one terrain scale, white start circle and goal star,
# green solid routes on free terrain and blue short dashes when obstacles are blocked.
from trajectory_style import render_cost_comparison, draw_comparison, comparison_handles  # noqa: E402, F401
from passage.plotting import decode_scene, render_scene, terrain_colormap, OBSTACLE_COLOR  # noqa: E402, F401


def first_scene(resolution, split, min_share=0.25, min_coverage=0.05):
    """Selection rule fixed in advance: the first scene of the split whose six stored routes all span at least
    `min_share` of the grid side and whose obstacles cover at least `min_coverage` of the cells."""
    for candidate in source.iter_rows(resolution, split, columns=["id", "metadata"], samples=False):
        lengths = [variant["length"] for paths in candidate["metadata"]["paths"].values()
                   for variant in (paths["free"], paths["obstacles"])]
        coverage = candidate["metadata"]["obstacles"]["ratio_actual"]
        if min(lengths) >= min_share * resolution and coverage >= min_coverage:
            return candidate["id"]
    raise ValueError("No scene of the split meets the selection rule")

## 1. What a planner receives

The tensor has shape (N, N, 3). Channel 0 is the elevation, normalised with the global calibration
range of the release; multiplying back gives metres. Channel 1 marks the start (0) and the goal (1)
on a background of -1. Channel 2 is the obstacle mask, with 1 for a blocked cell.

In [ ]:
row = source.get_row(resolution, split, sample_id=sample_id or first_scene(resolution, split), samples=False)
elevation, obstacles, start, goal = decode_scene(row)
calibration = row["metadata"]["calibration"]

fig, axes = plt.subplots(1, 3, figsize=(12, 4.3))
fig.subplots_adjust(wspace=0.42, bottom=0.14)
image = axes[0].imshow(elevation, cmap=terrain_colormap(), interpolation="nearest")
fig.colorbar(image, ax=axes[0], fraction=0.046, pad=0.03).set_label("Elevation (m)")
axes[0].set_title("Channel 0 · elevation")
axes[1].imshow(np.zeros(elevation.shape), cmap=plt.matplotlib.colors.ListedColormap(["#f4f3ee"]))
axes[1].scatter([start[1]], [start[0]], s=70, marker="o", facecolor="white", edgecolor=INK, zorder=3, label="Start (0)")
axes[1].scatter([goal[1]], [goal[0]], s=120, marker="*", facecolor="white", edgecolor=INK, zorder=3, label="Goal (1)")
axes[1].set_title("Channel 1 · markers")
axes[1].legend(loc="lower center", bbox_to_anchor=(0.5, -0.3), ncol=2, fontsize=8)
axes[2].imshow(obstacles, cmap=plt.matplotlib.colors.ListedColormap(["#f4f3ee", OBSTACLE_COLOR]),
               interpolation="nearest")
axes[2].set_title(f"Channel 2 · obstacles ({100 * obstacles.mean():.1f} % blocked)")
for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)
figure_title(fig, f"Scene {row['id']} · {resolution}×{resolution} · {split}",
             f"Calibration range {calibration['min']:,.0f} m to {calibration['max']:,.0f} m · "
             f"start {tuple(int(v) for v in start)} · goal {tuple(int(v) for v in goal)} (row, column)")
figures = save_figure(fig, output_path, f"display-{resolution}-{row['id']}-channels")
plt.show()

*Figure 1. The three input channels of one scene. Elevation is shown in metres after removing the
normalisation. Large grids are shown at full resolution here; individual cells may be smaller than a
screen pixel.*

## 2. The six reference routes

Figure 2 keeps the three objectives in columns and the two conditions in rows. Every panel uses the
same elevation scale. Following the paper's style, routes on free terrain are green and solid, and
routes that respect the obstacles are blue with short dashes. The white circle is the start and the
white star the goal.

In [ ]:
board = render_cost_comparison(row, title=f"Stored routes · scene {row['id']} · {resolution}×{resolution} · {split}")
figures += save_figure(board, output_path, f"display-{resolution}-{row['id']}")
plt.show()

*Figure 2. The six stored routes of the scene. Columns: elevation, uphill slope and energy objectives.
Top row: free terrain. Bottom row: obstacles blocked. Each panel states the number of route cells.
Grids larger than 512 cells per side are drawn on a 512-cell preview; route coordinates stay native.*

## 3. Length and cost of each route

The evaluator recomputes each route's cost from its cells, with the step costs of the dataset card.
The recomputed cost must equal the stored one. Costs are in metre-based units of each objective, so
compare them within an objective, not across objectives.

In [ ]:
from passage.notebook_reports import route_metrics_report

metrics = route_metrics_report(row, resolution, split)
routes = pd.DataFrame([{
    "Objective": OBJECTIVE_LABELS.get(r["cost_model"], r["cost_model"]), "Condition": r["task_variant"],
    "Status": r["stored_status"], "Cells": r["vertices"], "Stored cost": r["stored_cost"],
    "Recomputed cost": r["canonical_cost"], "Costs agree": r["cost_agreement"], "Legal": r["legal"],
} for r in metrics["routes"]])
show(routes, "Table 1. Stored routes of the scene and their recomputed costs.", "{:,.1f}")

## Summary

In [ ]:
free = routes[routes.Condition == "free"].set_index("Objective")["Stored cost"]
blocked = routes[routes.Condition == "obstacles"].set_index("Objective")["Stored cost"]
detour = 100 * (blocked / free - 1)
display(Markdown("\n".join([
    f"- Scene {row['id']} spans {elevation.max() - elevation.min():,.0f} m of relief; obstacles block "
    f"{100 * obstacles.mean():.1f} % of its cells.",
    f"- All {int(routes.Legal.sum())} of {len(routes)} stored routes are legal, and "
    f"{int(routes['Costs agree'].sum())} of {len(routes)} recomputed costs equal the stored cost.",
    "- Blocking the obstacles raises the optimal cost by " + ", ".join(
        f"{value:.1f} % ({name.lower()})" for name, value in detour.items()) + ".",
])))
report = {"resolution": resolution, "split": split, "sample_id": row["id"],
          "routes": routes.to_dict("records"), "figures": [relative(f) for f in figures]}

In [ ]:
report_file = write_report(report, output_path, f"display-{resolution}.json")
display(Markdown(f"The numbers of this run are saved in `{relative(report_file)}`, next to the figures."))

---
*MIT License · Copyright (c) 2026 THALES. The full licence text is in the `LICENSE` file of the repository.*